## module 1: baseline classifier
plain resnet50 fine-tune — no bells and whistles. **kept as a fixed baseline**; the improved model lives in 04. do not add upgrades here, it is the comparison point.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image

In [ ]:
# cuda if a gpu-enabled torch build is present, else cpu
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
# subset toggle: set N_SPECIES to an int (e.g. 100) for a quick proof-of-concept run, None for all
N_SPECIES = None

# splits.csv from notebook 02: path, species, split
splits = pd.read_csv("../data/splits.csv")
if N_SPECIES:
    keep = sorted(splits["species"].unique())[:N_SPECIES]
    splits = splits[splits["species"].isin(keep)].reset_index(drop=True)

labels = sorted(splits["species"].unique())
label2idx = {s:i for i,s in enumerate(labels)}

print(len(labels), "species")
print(splits["split"].value_counts().to_string())

In [ ]:
# imagenet normalization, light aug on train
MEAN = [0.485,0.456,0.406]
STD = [0.229,0.224,0.225]

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(224,scale=(0.7,1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(MEAN,STD),
])
eval_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(MEAN,STD),
])

In [ ]:
class PlantSet(Dataset):
    def __init__(self, df, tf):
        self.df = df.reset_index(drop=True)
        self.tf = tf
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row["path"]).convert("RGB")
        return self.tf(img), label2idx[row["species"]]

train_ds = PlantSet(splits[splits["split"]=="train"], train_tf)
val_ds = PlantSet(splits[splits["split"]=="val"], eval_tf)
test_ds = PlantSet(splits[splits["split"]=="test"], eval_tf)

# batch 16 fits a 4gb gpu (resnet50 @224); bump if you have more vram
# num_workers=0 keeps it portable (windows spawn issues); bump on linux/mac
train_dl = DataLoader(train_ds,batch_size=16,shuffle=True,num_workers=0)
val_dl = DataLoader(val_ds,batch_size=16,shuffle=False,num_workers=0)
test_dl = DataLoader(test_ds,batch_size=16,shuffle=False,num_workers=0)
len(train_ds), len(val_ds), len(test_ds)

In [ ]:
# resnet50 pretrained, swap the head for our classes
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
model.fc = nn.Linear(model.fc.in_features, len(labels))
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

In [ ]:
def run_epoch(dl, train):
    model.train() if train else model.eval()
    total_loss, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for x,y in dl:
            x,y = x.to(device), y.to(device)
            out = model(x)
            loss = criterion(out,y)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item()*len(y)
            correct += (out.argmax(1)==y).sum().item()
            n += len(y)
    return total_loss/n, correct/n

In [ ]:
EPOCHS = 5
hist = {"train_loss":[],"val_loss":[],"train_acc":[],"val_acc":[]}

for ep in range(EPOCHS):
    tl,ta = run_epoch(train_dl, True)
    vl,va = run_epoch(val_dl, False)
    hist["train_loss"].append(tl); hist["val_loss"].append(vl)
    hist["train_acc"].append(ta); hist["val_acc"].append(va)
    print(f"epoch {ep+1}: train_loss {tl:.3f} acc {ta:.3f} | val_loss {vl:.3f} acc {va:.3f}")

In [ ]:
fig,axs = plt.subplots(1,2,figsize=(12,4))
fig.set_facecolor("linen")

axs[0].plot(hist["train_loss"],color="darkorange",label="train")
axs[0].plot(hist["val_loss"],color="sienna",label="val")
axs[0].set_xlabel("Epoch")
axs[0].set_ylabel("Loss")
axs[0].set_title("Loss",weight="bold")
axs[0].legend()

axs[1].plot(hist["train_acc"],color="darkorange",label="train")
axs[1].plot(hist["val_acc"],color="sienna",label="val")
axs[1].set_xlabel("Epoch")
axs[1].set_ylabel("Accuracy")
axs[1].set_title("Accuracy",weight="bold")
axs[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# top-1 / top-5 on test
model.eval()
top1, top5, n = 0, 0, 0
with torch.no_grad():
    for x,y in test_dl:
        x,y = x.to(device), y.to(device)
        out = model(x)
        top5_pred = out.topk(5,dim=1).indices
        top1 += (out.argmax(1)==y).sum().item()
        top5 += (top5_pred==y.unsqueeze(1)).any(1).sum().item()
        n += len(y)

print(f"test top-1: {top1/n:.3f}")
print(f"test top-5: {top5/n:.3f}")

In [ ]:
torch.save({"state_dict":model.state_dict(),"labels":labels}, "../checkpoints/resnet50_baseline.pt")
print("saved")